In [0]:

dbutils.widgets.removeAll()

In [0]:
# Celda 2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import current_timestamp, col

In [0]:
# Celda 3
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlaiza082026")

In [0]:
# Celda 4
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/amazon_bestsellers/bestsellers_with_categories.csv"

In [0]:
# Celda 5
bestsellers_schema = StructType(fields=[
    StructField("Name", StringType(), True),
    StructField("Author", StringType(), True),
    StructField("User Rating", DoubleType(), True),
    StructField("Reviews", IntegerType(), True),
    StructField("Price", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Genre", StringType(), True)
])

In [0]:
# Celda 6
df_bestsellers = spark.read \
    .option("header", True) \
    .schema(bestsellers_schema) \
    .csv(ruta)

In [0]:
bestsellers_renamed_df = df_bestsellers.select(
    col("Name").alias("name"),
    col("Author").alias("author"),
    col("User Rating").alias("user_rating"),
    col("Reviews").alias("reviews"),
    col("Price").alias("price"),
    col("Year").alias("year"),
    col("Genre").alias("genre")
)

In [0]:
# Celda 8
bestsellers_final_df = bestsellers_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 9
bestsellers_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.amazon_bestsellers")